In [1]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count
import numpy as np
import pandas as pd
import os

spark = SparkSession.builder \
    .appName("Pertemuan6-ParquetETL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

26/09/25 09:05:56 WARN Utils: Your hostname, tarin resolves to a loopback address: 127.0.1.1; using 192.168.1.18 instead (on interface wlp0s20f3)
26/09/25 09:05:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/25 09:05:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


In [3]:
df_transaksi = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
df_produk = spark.read.json("tugas6_produk.json")
df_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

print("Transaksi:", df_transaksi.count(), "baris")
df_transaksi.printSchema()

print("Produk:", df_produk.count(), "baris")
df_produk.printSchema()

print("Ulasan:", df_ulasan.count(), "baris")
df_ulasan.printSchema()

Transaksi: 5000 baris
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

Produk: 30 baris
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

Ulasan: 3500 baris
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



In [4]:
df_gabung = df_transaksi \
    .join(df_produk, on="product_id", how="inner") \
    .join(df_ulasan, on="order_id", how="left")

df_gabung = df_gabung.withColumn(
    "total_pendapatan", col("unit_terjual") * col("harga")
)

df_gabung.show(5)

+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
|order_id|product_id|unit_terjual|            tanggal| harga|    kategori|nama_produk|rating|total_pendapatan|
+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
|     TX0|        21|           2|2026-10-10 00:00:00| 25000|  Elektronik|  Produk-21|     1|           50000|
|     TX1|         8|           6|2026-10-25 00:00:00|250000|     Fashion|   Produk-8|  NULL|         1500000|
|     TX2|        25|           4|2026-10-13 00:00:00| 25000|  Elektronik|  Produk-25|     2|          100000|
|     TX3|         3|           4|2026-10-18 00:00:00| 75000|     Fashion|   Produk-3|     5|          300000|
|     TX4|        19|           6|2026-10-07 00:00:00| 75000|Rumah Tangga|  Produk-19|  NULL|          450000|
+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
o

In [5]:
df_gabung = df_gabung.withColumn(
    "ada_ulasan", col("rating").isNotNull()
)

df_gabung = df_gabung.na.fill({"rating": 0})

df_gabung.select("order_id", "product_id", "rating", "ada_ulasan").show(10)

+--------+----------+------+----------+
|order_id|product_id|rating|ada_ulasan|
+--------+----------+------+----------+
|     TX0|        21|     1|      true|
|     TX1|         8|     0|     false|
|     TX2|        25|     2|      true|
|     TX3|         3|     5|      true|
|     TX4|        19|     0|     false|
|     TX5|        10|     5|      true|
|     TX6|        26|     4|      true|
|     TX7|         4|     0|     false|
|     TX8|         7|     5|      true|
|     TX9|        30|     0|     false|
+--------+----------+------+----------+
only showing top 10 rows



In [9]:
!hdfs dfs -mkdir -p /tarin/home/Praktikum-BigData/Tugas6

df_gabung.write.mode("overwrite") \
    .partitionBy("kategori") \
    .parquet("hdfs://localhost:9000/tarin/home/Praktikum-BigData/Tugas6/hasil_etl")

print("Data tersimpan ke HDFS.")
!hdfs dfs -ls -R /tarin/home/Praktikum-BigData/Tugas6/hasil_etl

[Stage 28:>                                                         (0 + 1) / 1]

Data tersimpan ke HDFS.


-rw-r--r--   3 tarin supergroup          0 2026-09-25 09:07 /tarin/home/Praktikum-BigData/Tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - tarin supergroup          0 2026-09-25 09:07 /tarin/home/Praktikum-BigData/Tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 tarin supergroup      16364 2026-09-25 09:07 /tarin/home/Praktikum-BigData/Tugas6/hasil_etl/kategori=Elektronik/part-00000-f4c5298e-33d1-418c-b15a-2ce3d6d6aa02.c000.snappy.parquet
drwxr-xr-x   - tarin supergroup          0 2026-09-25 09:07 /tarin/home/Praktikum-BigData/Tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 tarin supergroup      11134 2026-09-25 09:07 /tarin/home/Praktikum-BigData/Tugas6/hasil_etl/kategori=Fashion/part-00000-f4c5298e-33d1-418c-b15a-2ce3d6d6aa02.c000.snappy.parquet
drwxr-xr-x   - tarin supergroup          0 2026-09-25 09:07 /tarin/home/Praktikum-BigData/Tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 tarin supergroup      10393 2026-09-25 09:07 /tarin/home/Praktikum-BigData/Tugas6/hasil_etl/kategori=Kese

In [10]:
df_final = spark.read.parquet("hdfs://localhost:9000/tarin/home/Praktikum-BigData/Tugas6/hasil_etl")
print("Jumlah baris hasil akhir:", df_final.count())

Jumlah baris hasil akhir: 5000


In [11]:
df_insight = df_final.groupBy("kategori").agg(
    count("order_id").alias("total_transaksi"),
    avg(col("ada_ulasan").cast("int")).alias("persentase_ada_ulasan")
).orderBy("persentase_ada_ulasan")

df_insight.show()

+------------+---------------+---------------------+
|    kategori|total_transaksi|persentase_ada_ulasan|
+------------+---------------+---------------------+
|     Makanan|            536|   0.6884328358208955|
|   Kesehatan|            961|   0.6888657648283039|
|     Fashion|           1035|   0.7014492753623188|
|Rumah Tangga|            815|   0.7055214723926381|
|  Elektronik|           1653|   0.7065940713853599|
+------------+---------------+---------------------+

